<a href="https://colab.research.google.com/github/vinod-hn/Cyberbullying-detection/blob/main/Cyberbullying_Detection_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1️⃣ Setup Environment

In [1]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

/bin/bash: line 1: nvidia-smi: command not found

✅ PyTorch version: 2.9.0+cpu
✅ CUDA available: False


In [2]:
# Mount Google Drive for saving models
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
!mkdir -p /content/drive/MyDrive/CyberbullyingDetection/models

Mounted at /content/drive


In [ ]:
# Clone the repository
!rm -rf /content/Cyberbullying-detection
!git clone https://github.com/vinod-hn/Cyberbullying-detection.git
%cd /content/Cyberbullying-detection
!git lfs pull  # Pull large model files if needed

Cloning into 'Cyberbullying-detection'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 169 (delta 13), reused 168 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 733.02 KiB | 5.51 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q torch torchvision torchaudio --upgrade
!pip install -q emoji indic-transliteration
!pip install -q sentencepiece protobuf

print("\n✅ All dependencies installed!")

## 2️⃣ Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add project root to path
PROJECT_ROOT = '/content/Cyberbullying-detection'
sys.path.insert(0, PROJECT_ROOT)

# Load datasets
DATA_PATH = os.path.join(PROJECT_ROOT, '00_data', 'processed')

train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_data.csv'))
val_df = pd.read_csv(os.path.join(DATA_PATH, 'val_data.csv'))
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_data.csv'))

print(f"📊 Dataset Sizes:")
print(f"   Train: {len(train_df):,} samples")
print(f"   Val:   {len(val_df):,} samples")
print(f"   Test:  {len(test_df):,} samples")
print(f"\n📋 Columns: {list(train_df.columns)}")
print(f"\n🔍 Sample data:")
train_df.head()

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
    label_col = 'label' if 'label' in df.columns else df.columns[-1]
    df[label_col].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
    ax.set_title(f'{name} Label Distribution')
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3️⃣ Preprocessing

In [ ]:
# Import preprocessing modules
from importlib import import_module

try:
    from preprocessing import text_normalizer, emoji_handler, slang_expander
    print("✅ Preprocessing modules loaded from package")
except ImportError:
    # Manual import if package structure differs
    sys.path.insert(0, os.path.join(PROJECT_ROOT, '01_preprocessing'))
    import text_normalizer
    import emoji_handler
    import slang_expander
    print("✅ Preprocessing modules loaded directly")

In [ ]:
# Simple preprocessing function
import re
import emoji

def preprocess_text(text):
    """Clean and normalize text for model input."""
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Convert emojis to text
    text = emoji.demojize(text, delimiters=(" ", " "))

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text.strip()

# Determine text column
text_col = 'text' if 'text' in train_df.columns else train_df.columns[0]
label_col = 'label' if 'label' in train_df.columns else train_df.columns[-1]

print(f"📝 Text column: {text_col}")
print(f"🏷️ Label column: {label_col}")

# Apply preprocessing
train_df['processed_text'] = train_df[text_col].apply(preprocess_text)
val_df['processed_text'] = val_df[text_col].apply(preprocess_text)
test_df['processed_text'] = test_df[text_col].apply(preprocess_text)

print(f"\n✅ Preprocessing complete!")
print(f"\n🔍 Sample processed text:")
print(train_df[['processed_text', label_col]].head())

## 4️⃣ Baseline Models (TF-IDF + SVM/Naive Bayes)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import pickle

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(train_df[label_col])
y_val = le.transform(val_df[label_col])
y_test = le.transform(test_df[label_col])

print(f"📋 Classes: {le.classes_}")

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(train_df['processed_text'])
X_val_tfidf = tfidf.transform(val_df['processed_text'])
X_test_tfidf = tfidf.transform(test_df['processed_text'])

print(f"✅ TF-IDF features: {X_train_tfidf.shape[1]:,}")

In [ ]:
# Train baseline models with checkpoints
import os

BASELINE_CHECKPOINT_PATH = '/content/drive/MyDrive/CyberbullyingDetection/checkpoints/baseline'
os.makedirs(BASELINE_CHECKPOINT_PATH, exist_ok=True)

baseline_models = {
    'Naive Bayes': MultinomialNB(),
    'SVM': SVC(kernel='linear', probability=True),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

baseline_results = {}

for name, model in baseline_models.items():
    checkpoint_file = f'{BASELINE_CHECKPOINT_PATH}/{name.replace(" ", "_").lower()}_checkpoint.pkl'
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"\n📂 Loading {name} from checkpoint...")
        with open(checkpoint_file, 'rb') as f:
            checkpoint_data = pickle.load(f)
            model = checkpoint_data['model']
            baseline_models[name] = model
        print(f"   ✅ {name} loaded from checkpoint!")
    else:
        print(f"\n🔄 Training {name}...")
        model.fit(X_train_tfidf, y_train)
        
        # Save checkpoint
        with open(checkpoint_file, 'wb') as f:
            pickle.dump({'model': model, 'name': name}, f)
        print(f"   💾 Checkpoint saved: {checkpoint_file}")

    # Evaluate on validation set
    y_pred = model.predict(X_val_tfidf)
    acc = accuracy_score(y_val, y_pred)
    baseline_results[name] = {'model': model, 'accuracy': acc}

    print(f"   ✅ {name} Validation Accuracy: {acc:.4f}")
    print(classification_report(y_val, y_pred, target_names=le.classes_))

print(f"\n✅ All baseline models trained with checkpoints!")

In [ ]:
# Save best baseline model
best_baseline = max(baseline_results.items(), key=lambda x: x[1]['accuracy'])
print(f"\n🏆 Best Baseline: {best_baseline[0]} (Accuracy: {best_baseline[1]['accuracy']:.4f})")

# Save to Drive
SAVE_PATH = '/content/drive/MyDrive/CyberbullyingDetection/models'

with open(f'{SAVE_PATH}/best_baseline_model.pkl', 'wb') as f:
    pickle.dump(best_baseline[1]['model'], f)

with open(f'{SAVE_PATH}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open(f'{SAVE_PATH}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print(f"✅ Baseline model saved to Google Drive!")

## 5️⃣ Transformer Model (BERT/mBERT) - GPU Training

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import torch
import gc

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Define 3 transformer models to train
TRANSFORMER_MODELS = {
    'BERT': 'bert-base-uncased',
    'mBERT': 'bert-base-multilingual-cased',
    'IndicBERT': 'ai4bharat/indic-bert'
}

print(f"\n📦 Models to train:")
for name, model_name in TRANSFORMER_MODELS.items():
    print(f"   • {name}: {model_name}")

In [ ]:
# Prepare base datasets for Transformers (will tokenize per model)
from datasets import Dataset

# Create base Hugging Face datasets
train_data = train_df[['processed_text', label_col]].rename(columns={'processed_text': 'text', label_col: 'label'})
val_data = val_df[['processed_text', label_col]].rename(columns={'processed_text': 'text', label_col: 'label'})
test_data = test_df[['processed_text', label_col]].rename(columns={'processed_text': 'text', label_col: 'label'})

# Convert labels to integers if they are strings
train_data['label'] = le.transform(train_data['label'])
val_data['label'] = le.transform(val_data['label'])
test_data['label'] = le.transform(test_data['label'])

print(f"✅ Base datasets prepared!")
print(f"   Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

# Create checkpoint directory for transformers
TRANSFORMER_CHECKPOINT_PATH = '/content/drive/MyDrive/CyberbullyingDetection/checkpoints/transformer'
os.makedirs(TRANSFORMER_CHECKPOINT_PATH, exist_ok=True)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def get_training_args(model_name):
    """Get training arguments optimized for Colab GPU with checkpointing."""
    checkpoint_dir = f'{TRANSFORMER_CHECKPOINT_PATH}/{model_name}'
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    return TrainingArguments(
        output_dir=checkpoint_dir,  # Save checkpoints to Google Drive
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=f'./logs_{model_name}',
        logging_steps=100,
        eval_strategy='steps',
        eval_steps=500,
        save_strategy='steps',
        save_steps=500,
        save_total_limit=3,  # Keep only last 3 checkpoints to save space
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        fp16=True,  # Mixed precision for faster GPU training
        gradient_accumulation_steps=2,
        dataloader_num_workers=2,
        report_to='none',
        resume_from_checkpoint=True  # Resume from checkpoint if available
    )

print("✅ Training configuration with checkpointing ready!")
print(f"📂 Transformer checkpoints will be saved to: {TRANSFORMER_CHECKPOINT_PATH}")

In [ ]:
# Train all 3 transformer models with checkpoints
transformer_results = {}
trained_models = {}

for model_key, model_path in TRANSFORMER_MODELS.items():
    print(f"\n{'='*60}")
    print(f"🚀 Training {model_key} ({model_path})")
    print(f"{'='*60}\n")
    
    checkpoint_dir = f'{TRANSFORMER_CHECKPOINT_PATH}/{model_key}'
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Check for existing checkpoint
        existing_checkpoints = []
        if os.path.exists(checkpoint_dir):
            existing_checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]
        
        if existing_checkpoints:
            # Find the latest checkpoint
            latest_checkpoint = max(existing_checkpoints, key=lambda x: int(x.split('-')[1]))
            latest_checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
            print(f"📂 Found checkpoint: {latest_checkpoint}")
            print(f"   Resuming training from: {latest_checkpoint_path}")
            
            # Load model from checkpoint
            model = AutoModelForSequenceClassification.from_pretrained(
                latest_checkpoint_path,
                num_labels=len(le.classes_)
            ).to(device)
        else:
            print(f"🆕 No checkpoint found. Starting fresh training...")
            # Load fresh model
            model = AutoModelForSequenceClassification.from_pretrained(
                model_path,
                num_labels=len(le.classes_)
            ).to(device)
        
        print(f"✅ Model loaded! Parameters: {model.num_parameters():,}")
        
        # Tokenize datasets
        def tokenize_function(examples):
            return tokenizer(
                examples['text'],
                padding='max_length',
                truncation=True,
                max_length=128
            )
        
        # Create and tokenize datasets
        train_dataset = Dataset.from_pandas(train_data)
        val_dataset = Dataset.from_pandas(val_data)
        test_dataset = Dataset.from_pandas(test_data)
        
        train_dataset = train_dataset.map(tokenize_function, batched=True)
        val_dataset = val_dataset.map(tokenize_function, batched=True)
        test_dataset = test_dataset.map(tokenize_function, batched=True)
        
        # Set format for PyTorch
        train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
        val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
        test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
        
        # Create Trainer with checkpoint support
        training_args = get_training_args(model_key)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        
        # Train (will resume from checkpoint if available)
        print(f"🔄 Training {model_key} with GPU acceleration...")
        resume_checkpoint = None
        if existing_checkpoints:
            resume_checkpoint = os.path.join(checkpoint_dir, latest_checkpoint)
        
        train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
        
        print(f"\n✅ {model_key} Training complete!")
        print(f"   Total steps: {train_result.global_step}")
        print(f"   Training loss: {train_result.training_loss:.4f}")
        print(f"   💾 Checkpoints saved to: {checkpoint_dir}")
        
        # Evaluate on test set
        print(f"\n📊 Evaluating {model_key} on test set...")
        test_results = trainer.evaluate(test_dataset)
        
        # Get predictions
        predictions = trainer.predict(test_dataset)
        preds = predictions.predictions.argmax(-1)
        
        # Store results
        transformer_results[model_key] = {
            'test_results': test_results,
            'predictions': predictions,
            'accuracy': test_results['eval_accuracy'],
            'f1': test_results['eval_f1']
        }
        
        trained_models[model_key] = {
            'model': model,
            'tokenizer': tokenizer,
            'trainer': trainer,
            'test_dataset': test_dataset
        }
        
        print(f"\n📈 {model_key} Test Results:")
        print(f"   Accuracy: {test_results['eval_accuracy']:.4f}")
        print(f"   F1 Score: {test_results['eval_f1']:.4f}")
        print(f"   Precision: {test_results['eval_precision']:.4f}")
        print(f"   Recall: {test_results['eval_recall']:.4f}")
        
        print("\n📋 Classification Report:")
        print(classification_report(test_dataset['label'], preds, target_names=le.classes_))
        
        # Clear GPU memory for next model
        if model_key != list(TRANSFORMER_MODELS.keys())[-1]:
            del model, trainer
            gc.collect()
            torch.cuda.empty_cache()
            print(f"🧹 GPU memory cleared for next model\n")
            
    except Exception as e:
        print(f"❌ Error training {model_key}: {str(e)}")
        transformer_results[model_key] = {'error': str(e)}
        continue

print(f"\n{'='*60}")
print("✅ All transformer models trained with checkpoints!")
print(f"📂 Checkpoints saved to: {TRANSFORMER_CHECKPOINT_PATH}")
print(f"{'='*60}")

In [ ]:
# Compare all transformer models
print("📊 Transformer Model Comparison:\n")
print(f"{'Model':<15} {'Accuracy':<12} {'F1 Score':<12} {'Precision':<12} {'Recall':<12}")
print("-" * 63)

for model_key, results in transformer_results.items():
    if 'error' not in results:
        print(f"{model_key:<15} {results['test_results']['eval_accuracy']:<12.4f} "
              f"{results['test_results']['eval_f1']:<12.4f} "
              f"{results['test_results']['eval_precision']:<12.4f} "
              f"{results['test_results']['eval_recall']:<12.4f}")
    else:
        print(f"{model_key:<15} {'ERROR':<12} - {results['error'][:30]}...")

# Find best transformer model
valid_results = {k: v for k, v in transformer_results.items() if 'error' not in v}
if valid_results:
    best_transformer_name = max(valid_results.keys(), key=lambda x: valid_results[x]['f1'])
    best_transformer = trained_models[best_transformer_name]
    
    print(f"\n🏆 Best Transformer Model: {best_transformer_name}")
    print(f"   F1 Score: {transformer_results[best_transformer_name]['f1']:.4f}")
    print(f"   Accuracy: {transformer_results[best_transformer_name]['accuracy']:.4f}")
else:
    print("\n❌ No transformer models trained successfully")

In [ ]:
# Save all transformer models to Google Drive
import json

for model_key in trained_models.keys():
    if 'error' not in transformer_results[model_key]:
        model_save_path = f'{SAVE_PATH}/transformer_{model_key.lower()}'
        
        trained_models[model_key]['trainer'].save_model(model_save_path)
        trained_models[model_key]['tokenizer'].save_pretrained(model_save_path)
        
        # Save test metrics
        with open(f'{model_save_path}/test_metrics.json', 'w') as f:
            json.dump({k: float(v) if isinstance(v, (int, float)) else v 
                      for k, v in transformer_results[model_key]['test_results'].items()}, f, indent=2)
        
        print(f"✅ {model_key} model saved to: {model_save_path}")

# Save comparison summary
comparison_summary = {
    'models_trained': list(TRANSFORMER_MODELS.keys()),
    'best_model': best_transformer_name if valid_results else None,
    'results': {k: {'accuracy': v['accuracy'], 'f1': v['f1']} 
                for k, v in transformer_results.items() if 'error' not in v}
}

with open(f'{SAVE_PATH}/transformer_comparison.json', 'w') as f:
    json.dump(comparison_summary, f, indent=2)

print(f"\n✅ All transformer models and comparison saved to Google Drive!")

# Set best model variables for ensemble section
model = best_transformer['model']
tokenizer = best_transformer['tokenizer']
predictions = transformer_results[best_transformer_name]['predictions']
test_results = transformer_results[best_transformer_name]['test_results']
test_dataset = best_transformer['test_dataset']
MODEL_NAME = TRANSFORMER_MODELS[best_transformer_name]

print(f"\n📌 Using {best_transformer_name} for ensemble model")

In [ ]:
# Visualize all models comparison
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Accuracy comparison (all models)
ax1 = axes[0, 0]
models = list(model_results.keys())
accuracies = list(model_results.values())

# Color coding: blue for baseline, different colors for each transformer
colors = []
for m in models:
    if 'BERT' in m:
        colors.append('#e74c3c')  # Red for BERT
    elif 'mBERT' in m:
        colors.append('#9b59b6')  # Purple for mBERT
    elif 'IndicBERT' in m:
        colors.append('#e67e22')  # Orange for IndicBERT
    else:
        colors.append('#3498db')  # Blue for baseline

bars = ax1.barh(models, accuracies, color=colors)
ax1.set_xlabel('Accuracy')
ax1.set_title('All Models - Accuracy Comparison')
ax1.set_xlim(0, 1)
for bar, acc in zip(bars, accuracies):
    ax1.text(acc + 0.01, bar.get_y() + bar.get_height()/2, f'{acc:.4f}', va='center')

# 2. F1 Score comparison (all models)
ax2 = axes[0, 1]
f1_scores = list(model_f1_scores.values())
bars = ax2.barh(models, f1_scores, color=colors)
ax2.set_xlabel('F1 Score')
ax2.set_title('All Models - F1 Score Comparison')
ax2.set_xlim(0, 1)
for bar, f1 in zip(bars, f1_scores):
    ax2.text(f1 + 0.01, bar.get_y() + bar.get_height()/2, f'{f1:.4f}', va='center')

# 3. Transformer models detailed comparison
ax3 = axes[1, 0]
valid_transformers = [k for k in transformer_results.keys() if 'error' not in transformer_results[k]]
metrics = ['accuracy', 'f1', 'precision', 'recall']
x = np.arange(len(valid_transformers))
width = 0.2

for i, metric in enumerate(metrics):
    values = [transformer_results[m]['test_results'][f'eval_{metric}'] for m in valid_transformers]
    ax3.bar(x + i*width, values, width, label=metric.capitalize())

ax3.set_xlabel('Model')
ax3.set_ylabel('Score')
ax3.set_title('Transformer Models - Detailed Metrics')
ax3.set_xticks(x + width * 1.5)
ax3.set_xticklabels(valid_transformers)
ax3.legend()
ax3.set_ylim(0, 1)

# 4. Confusion matrix for best transformer
ax4 = axes[1, 1]
if valid_transformers:
    best_transformer_name = max(valid_transformers, key=lambda x: transformer_results[x]['f1'])
    best_preds = transformer_results[best_transformer_name]['predictions'].predictions.argmax(-1)
    cm = confusion_matrix(y_test, best_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=ax4)
    ax4.set_title(f'Best Transformer ({best_transformer_name}) - Confusion Matrix')
    ax4.set_xlabel('Predicted')
    ax4.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Comparison chart saved to Google Drive!")

## 6️⃣ Model Comparison & Visualization

In [ ]:
# Compare all models (Baseline + All 3 Transformers)
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns

# Collect baseline results on test set
model_results = {}
model_f1_scores = {}

print("📊 Evaluating all models on test set...\n")

# Baseline models
print("🔹 Baseline Models:")
for name, data in baseline_results.items():
    y_pred = data['model'].predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    model_results[name] = acc
    model_f1_scores[name] = f1
    print(f"   {name}: Accuracy={acc:.4f}, F1={f1:.4f}")

# Transformer models (all 3)
print("\n🔹 Transformer Models:")
for model_key, results in transformer_results.items():
    if 'error' not in results:
        acc = results['accuracy']
        f1 = results['f1']
        model_results[f'Transformer ({model_key})'] = acc
        model_f1_scores[f'Transformer ({model_key})'] = f1
        print(f"   {model_key}: Accuracy={acc:.4f}, F1={f1:.4f}")
    else:
        print(f"   {model_key}: ERROR - {results['error'][:50]}...")

# Find best overall model
best_model_name = max(model_f1_scores.keys(), key=lambda x: model_f1_scores[x])
print(f"\n🏆 Best Overall Model: {best_model_name}")
print(f"   F1 Score: {model_f1_scores[best_model_name]:.4f}")
print(f"   Accuracy: {model_results[best_model_name]:.4f}")

## 7️⃣ Inference Example

In [ ]:
# Use best transformer model for inference
from scipy.special import softmax

# Get best transformer model
best_transformer_name = max(
    [k for k in transformer_results.keys() if 'error' not in transformer_results[k]], 
    key=lambda x: transformer_results[x]['f1']
)
best_model = trained_models[best_transformer_name]['model']
best_tokenizer = trained_models[best_transformer_name]['tokenizer']

print(f"🎯 Using best model for inference: {best_transformer_name}")
print(f"   F1 Score: {transformer_results[best_transformer_name]['f1']:.4f}\n")

def predict_cyberbullying(text):
    """Predict if text is cyberbullying using the best transformer model."""
    # Preprocess
    processed = preprocess_text(text)

    # Transformer prediction
    inputs = best_tokenizer(processed, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = best_model(**inputs)
    probs = softmax(outputs.logits.cpu().numpy(), axis=1)
    
    prediction = le.inverse_transform([probs.argmax()])[0]
    confidence = probs.max()

    return {
        'text': text,
        'prediction': prediction,
        'confidence': float(confidence),
        'probabilities': {cls: float(p) for cls, p in zip(le.classes_, probs[0])}
    }

# Test examples
test_texts = [
    "You're such a loser, nobody likes you!",
    "Great job on the presentation today!",
    "I hate you so much, go die!",
    "Thanks for helping me with the project 😊"
]

print("🔮 Inference Examples:\n")
for text in test_texts:
    result = predict_cyberbullying(text)
    emoji_icon = "🚨" if result['prediction'] != 'Not Cyberbullying' else "✅"
    print(f"{emoji_icon} Text: \"{text}\"")
    print(f"   Prediction: {result['prediction']} (Confidence: {result['confidence']:.2%})")
    print()

## 8️⃣ Download Models

In [ ]:
# Create zip of all models for download
import shutil

# Zip the models folder
!cd /content/drive/MyDrive/CyberbullyingDetection && zip -r /content/cyberbullying_models.zip models/

# Download
from google.colab import files
files.download('/content/cyberbullying_models.zip')

print("\n✅ Models downloaded! You can also find them in your Google Drive.")

## 📋 Summary

### Models Trained:
1. **Baseline Models** (TF-IDF) with Checkpoints:
   - Naive Bayes
   - SVM
   - Logistic Regression

2. **Transformer Models** (GPU-accelerated with Checkpoints):
   - BERT (`bert-base-uncased`)
   - mBERT (`bert-base-multilingual-cased`)
   - IndicBERT (`ai4bharat/indic-bert`)

### Checkpointing:
- Baseline checkpoints saved to: `checkpoints/baseline/`
- Transformer checkpoints saved to: `checkpoints/transformer/`
- Training can be resumed from checkpoints if interrupted

### Files Saved to Google Drive:
- `models/best_baseline_model.pkl`
- `models/tfidf_vectorizer.pkl`
- `models/label_encoder.pkl`
- `models/transformer_bert/` (BERT model + tokenizer)
- `models/transformer_mbert/` (mBERT model + tokenizer)
- `models/transformer_indicbert/` (IndicBERT model + tokenizer)
- `models/transformer_comparison.json`
- `models/all_models_comparison.png`

### Next Steps:
1. Download models and integrate into your project
2. Fine-tune hyperparameters for better performance
3. Use the best performing model for production
4. Deploy as API using the 06_api module